# 13. Sevilla Dish Candidate Detection

Este notebook construye la primera detección seria de **candidatos de platos** sobre las reviews reales de Google Places Sevilla exportadas desde PostgreSQL.

El objetivo no es todavía generar el ranking final, sino crear una capa intermedia fiable:

```text
reviews reales de Sevilla
→ menciones candidatas de platos
→ clasificación inicial
→ artefactos revisables
→ base para normalización, sentimiento y ranking sevilla_pilot
```

La estrategia inicial es híbrida y explicable:

- lexicón gastronómico español/andaluz;
- patrones para platos compuestos;
- clasificación de términos genéricos, bebidas e ingredientes;
- contexto local de la mención;
- puntuación de confianza;
- exportación para revisión manual.

No se usa GPU ni modelos pesados en esta fase.


In [2]:
# ============================================================
# 01. Imports y configuración general
# ============================================================

import json
import re
import hashlib
import unicodedata
from pathlib import Path
from collections import Counter, defaultdict
from datetime import datetime, timezone

import pandas as pd

try:
    from IPython.display import display
except Exception:
    display = print


pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 220)
pd.set_option("display.width", 160)

# Resolver raíz del proyecto de forma flexible buscando una carpeta data/ hacia arriba.
def find_project_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "data").exists():
            return candidate
    return cwd

PROJECT_DIR = find_project_dir()

INPUT_PATH = PROJECT_DIR / "data" / "artifacts" / "ai" / "sevilla" / "reviews_for_ai_google_places.jsonl"
OUTPUT_DIR = PROJECT_DIR / "data" / "artifacts" / "ai" / "sevilla" / "dish_detection"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RUN_NAME = "sevilla_dish_candidate_detection_v1"
GENERATED_AT = datetime.now(timezone.utc).isoformat()

print("PROJECT_DIR:", PROJECT_DIR)
print("INPUT_PATH:", INPUT_PATH)
print("OUTPUT_DIR:", OUTPUT_DIR)


PROJECT_DIR: C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline
INPUT_PATH: C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\reviews_for_ai_google_places.jsonl
OUTPUT_DIR: C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_detection


In [3]:
# ============================================================
# 02. Carga del JSONL exportado para IA
# ============================================================

def read_jsonl(path: Path):
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line_number, line in enumerate(f, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f"JSON inválido en línea {line_number}: {exc}") from exc
    return rows

if not INPUT_PATH.exists():
    raise FileNotFoundError(f"No existe el archivo de entrada: {INPUT_PATH}")

reviews = read_jsonl(INPUT_PATH)
df = pd.DataFrame(reviews)

print("Reviews cargadas:", len(df))
print("Columnas:", list(df.columns))
display(df.head(3))


Reviews cargadas: 4110
Columnas: ['review_id', 'place_id', 'place_source_ref_id', 'source_system_code', 'source_review_id', 'source_place_record_id', 'place_source_record_id', 'place_name', 'place_canonical_name', 'address_text', 'city', 'neighborhood_id', 'neighborhood_name', 'district_id', 'district_name', 'rating_value', 'review_language', 'review_created_at', 'review_updated_at', 'text', 'text_normalized', 'is_operational_review', 'is_training_eligible', 'source_review_url', 'place_latitude', 'place_longitude', 'text_length_chars']


,review_id,place_id,place_source_ref_id,source_system_code,source_review_id,source_place_record_id,place_source_record_id,place_name,place_canonical_name,address_text,city,neighborhood_id,neighborhood_name,district_id,district_name,rating_value,review_language,review_created_at,review_updated_at,text,text_normalized,is_operational_review,is_training_eligible,source_review_url,place_latitude,place_longitude,text_length_chars
0,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,google_places,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,Tropitali,"C. San Julián, 13, Casco Antiguo, 41003 Sevilla",Sevilla,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,5.0,es,2026-05-10T19:07:36.701337+02:00,2026-05-10T19:07:36.701337+02:00,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis ami...","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis ami...",True,True,https://www.google.com/maps/reviews/data=!4m6!14m5!1m4!2m3!1sCi9DQUlRQUNvZENodHljRjlvT21oaWFuRkNhV3RFYUc5TVFVNDJWVGRQZGxkbk5uYxAB!2m1!1s0xd126d7e02e048e7:0xe1f1c89e68a42407,37.399171,-5.985623,221
1,b680ad6e-8999-48a0-a754-84ea920429ab,a6ba370c-eb3d-4456-94cf-58617d1a3b0a,9f4c2eb7-2ff7-44ce-8db2-1be259e6985d,google_places,4897b8c24f949dda32b4e6c5cad74f4a91228c64182c5ce9408158505e192ce0,ChIJuQj13EFvEg0RPkUdEUzL0BA,ChIJuQj13EFvEg0RPkUdEUzL0BA,Cariña VZ,Cariña VZ,"C. Pdte. Cárdenas, 10, 41013 Sevilla",Sevilla,65af0982-2885-49df-9f09-8b784c2f3d7b,GIRALDA SUR,22b433e4-3b14-4b09-bfaa-f0ca1d7e6bdc,Sur,5.0,es,2026-05-10T18:17:25.948671+02:00,2026-05-10T18:17:25.948671+02:00,"Buena comida y con un excelente servicio de Leo que nos atendió y nos explicó todo muy bien, sin duda iremos más de una vez.","Buena comida y con un excelente servicio de Leo que nos atendió y nos explicó todo muy bien, sin duda iremos más de una vez.",True,True,https://www.google.com/maps/reviews/data=!4m6!14m5!1m4!2m3!1sCi9DQUlRQUNvZENodHljRjlvT2sxVGNtOXFka2xKV0daTVNrZFJWRGhZUm1OMVpHYxAB!2m1!1s0xd126f41dcf508b9:0x10d0cb4c111d453e,37.373058,-5.977433,124
2,b2ac02e7-4348-481d-ad2e-cb37ab9c2685,7d39aad6-1d3f-4fc8-8d4b-22896354507e,4fa65ed3-e51a-4a96-b82c-cbe002c03141,google_places,c19ee9e9030a22bbcd80d00ef0680365a36e284ccb72955b931c805b5c37aade,ChIJmU6FwYdvEg0RYLF2daPZQkc,ChIJmU6FwYdvEg0RYLF2daPZQkc,TOTORA,TOTORA,"Av. los Gavilanes, 89, 41006 Sevilla",Sevilla,76a474aa-a390-48b4-a7ad-d68f6d747832,LA PLATA,09b2875d-a826-4c21-bd78-1d8fa48009de,Cerro - Amate,2.0,es,2026-05-10T18:03:34.257906+02:00,2026-05-10T18:03:34.257906+02:00,"Demasiada tiempo de espera para recibir mi pedido\nEl ceviche muy ácido , lo rescatable los anticuchos10/10","Demasiada tiempo de espera para recibir mi pedido El ceviche muy ácido , lo rescatable los anticuchos10/10",True,True,https://www.google.com/maps/reviews/data=!4m6!14m5!1m4!2m3!1sCi9DQUlRQUNvZENodHljRjlvT25Oc2MzRTNhamx2Y0hoa2JqRk5ZekJGUm01dFdYYxAB!2m1!1s0xd126f87c1854e99:0x4742d9a37576b160,37.371563,-5.951597,106


In [4]:
# ============================================================
# 03. Preparación de campos auxiliares
# ============================================================

REQUIRED_COLUMNS = [
    "review_id",
    "place_id",
    "place_source_ref_id",
    "source_system_code",
    "source_review_id",
    "source_place_record_id",
    "place_name",
    "district_name",
    "neighborhood_name",
    "rating_value",
    "review_language",
    "text",
]

missing_columns = [col for col in REQUIRED_COLUMNS if col not in df.columns]
if missing_columns:
    raise ValueError(f"Faltan columnas obligatorias: {missing_columns}")

def rating_to_sentiment(rating):
    try:
        rating = float(rating)
    except Exception:
        return "unknown"
    if rating >= 4:
        return "positive"
    if rating <= 2:
        return "negative"
    return "neutral"

# Texto principal.
df["text"] = df["text"].fillna("").astype(str)
df["text_clean"] = df["text"].str.replace(r"\s+", " ", regex=True).str.strip()
df["text_length_chars"] = df["text_clean"].str.len()
df["text_length_words"] = df["text_clean"].str.split().apply(len)
df["review_sentiment_from_rating"] = df["rating_value"].apply(rating_to_sentiment)

# Mantener preferentemente español para el prototipo, pero sin borrar los demás de entrada.
display(df[["review_language", "rating_value", "review_sentiment_from_rating", "text_length_chars"]].describe(include="all"))
print("Idiomas:")
print(df["review_language"].value_counts(dropna=False))
print("Ratings:")
print(df["rating_value"].value_counts(dropna=False).sort_index())


,review_language,rating_value,review_sentiment_from_rating,text_length_chars
count,4110,4110.000000,4110,4110.000000
unique,3,NaN,3,NaN
top,es,NaN,positive,NaN
freq,4108,NaN,3424,NaN
mean,NaN,4.325061,NaN,353.586861
std,NaN,1.234859,NaN,282.810384
min,NaN,1.000000,NaN,24.000000
25%,NaN,4.000000,NaN,177.000000
50%,NaN,5.000000,NaN,282.000000
75%,NaN,5.000000,NaN,433.000000


Idiomas:
review_language
es    4108
fr       1
pt       1
Name: count, dtype: int64
Ratings:
rating_value
1.0     348
2.0     128
3.0     210
4.0     578
5.0    2846
Name: count, dtype: int64


## 04. Normalización de texto

Para detectar candidatos de forma robusta, usamos una normalización para matching que:

- convierte a minúsculas;
- elimina tildes;
- mantiene la longitud aproximada del texto para poder recuperar spans del texto original.

Esto permite detectar `jamón` como `jamon`, `atún` como `atun`, etc.


In [5]:
# ============================================================
# 04. Normalización para matching y utilidades de contexto
# ============================================================

def normalize_for_match_keep_len(text: str) -> str:
    """Normaliza a minúsculas y elimina tildes intentando mantener 1 char por char original."""
    out = []
    for ch in str(text):
        decomposed = unicodedata.normalize("NFD", ch)
        base = "".join(c for c in decomposed if unicodedata.category(c) != "Mn")
        if not base:
            base = ch
        # Mantener un solo carácter para preservar offsets en la mayoría de casos.
        out.append(base[0].lower())
    return "".join(out)

def normalize_text_value(text: str) -> str:
    text = normalize_for_match_keep_len(text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def compact_spaces(text: str) -> str:
    return re.sub(r"\s+", " ", str(text)).strip()

def get_sentence_context(text: str, start: int, end: int) -> str:
    """Devuelve la oración aproximada que contiene el span."""
    text = str(text)
    for match in re.finditer(r"[^.!?\n]+[.!?]*", text):
        if match.start() <= start < match.end():
            return compact_spaces(match.group(0))
    # Fallback: ventana local.
    left = max(0, start - 140)
    right = min(len(text), end + 140)
    return compact_spaces(text[left:right])

def get_window_context(text: str, start: int, end: int, window: int = 120) -> str:
    left = max(0, start - window)
    right = min(len(text), end + window)
    return compact_spaces(text[left:right])

def stable_id(*parts) -> str:
    raw = "||".join("" if p is None else str(p) for p in parts)
    return hashlib.sha1(raw.encode("utf-8")).hexdigest()

# Prueba rápida.
example = "Pedimos croquetas de jamón, solomillo al whisky y una tarta de queso."
print(normalize_for_match_keep_len(example))
print(get_sentence_context(example, 8, 27))


pedimos croquetas de jamon, solomillo al whisky y una tarta de queso.
Pedimos croquetas de jamón, solomillo al whisky y una tarta de queso.


## 05. Lexicón gastronómico inicial

Se separan cuatro grupos:

1. `dish_core`: platos o formatos que pueden funcionar como candidatos de plato.
2. `generic_food_term`: términos demasiado genéricos para ranking directo.
3. `beverage`: bebidas, excluidas del ranking de platos.
4. `ingredient_or_side`: ingredientes o acompañamientos que solo serán útiles si aparecen en compuestos.

Esta separación evita que el ranking acabe recomendando `cerveza`, `pan`, `tapas` o `comida` como si fueran platos concretos.


In [6]:
# ============================================================
# 05. Lexicón gastronómico inicial ES/Sevilla
# ============================================================

alias_records = []

def add_alias(alias, canonical=None, group="dish_core", priority=1.0):
    alias = compact_spaces(alias)
    canonical = compact_spaces(canonical or alias)
    alias_records.append({
        "alias": alias,
        "alias_norm": normalize_text_value(alias),
        "canonical": normalize_text_value(canonical),
        "group": group,
        "priority": priority,
    })

# Platos y formatos frecuentes en Sevilla / España.
DISH_ALIASES = {
    # Tapas/platos concretos tradicionales
    "croqueta": ["croqueta", "croquetas", "croquetas caseras", "croquetas de jamón", "croquetas de puchero", "croquetas de setas", "croquetas de boletus"],
    "ensaladilla": ["ensaladilla", "ensaladilla rusa"],
    "salmorejo": ["salmorejo"],
    "gazpacho": ["gazpacho"],
    "tortilla de patatas": ["tortilla", "tortilla de patatas", "tortilla española"],
    "patatas bravas": ["bravas", "patatas bravas", "papas bravas"],
    "patatas alioli": ["patatas alioli", "papas alioli", "patatas con alioli"],
    "huevos rotos": ["huevos rotos", "huevos estrellados"],
    "carrillada": ["carrillada", "carrillada ibérica", "carrillera", "carrilleras"],
    "solomillo al whisky": ["solomillo", "solomillo al whisky", "solomillo whisky"],
    "presa ibérica": ["presa", "presa ibérica", "presa iberica"],
    "secreto ibérico": ["secreto", "secreto ibérico", "secreto iberico"],
    "lagarto ibérico": ["lagarto", "lagarto ibérico", "lagarto iberico"],
    "lomo": ["lomo", "lomo al whisky"],
    "rabo de toro": ["rabo de toro", "cola de toro"],
    "pringá": ["pringá", "pringa", "montadito de pringá", "montadito de pringa"],
    "flamenquín": ["flamenquín", "flamenquin"],
    "albóndigas": ["albóndigas", "albondigas"],
    "chicharrones": ["chicharrones"],
    "morcilla": ["morcilla"],
    # Pescado/marisco
    "bacalao": ["bacalao", "bacalao dorado"],
    "atún": ["atún", "atun", "atún rojo", "atun rojo", "tataki de atún", "tataki de atun"],
    "gambas": ["gambas", "gambas al ajillo"],
    "langostinos": ["langostinos"],
    "pulpo": ["pulpo", "pulpo a la brasa", "pulpo a la gallega"],
    "calamares": ["calamares", "calamar"],
    "chocos": ["chocos", "choco"],
    "puntillitas": ["puntillitas"],
    "boquerones": ["boquerones"],
    "cazón en adobo": ["cazón en adobo", "cazon en adobo", "adobo"],
    "pescado frito": ["pescado frito", "fritura de pescado", "frito variado"],
    "mejillones": ["mejillones"],
    "almejas": ["almejas"],
    "navajas": ["navajas"],
    # Arroces/pastas/internacional
    "paella": ["paella"],
    "arroz": ["arroz", "arroz negro", "arroz con pollo", "arroz con leche"],
    "risotto": ["risotto"],
    "pasta": ["pasta", "ravioli", "gnocchi", "lasaña", "lasagna"],
    "pizza": ["pizza", "pizzas"],
    "hamburguesa": ["hamburguesa", "hamburguesas", "burger", "burguer"],
    "sushi": ["sushi"],
    "ramen": ["ramen"],
    "taco": ["taco", "tacos"],
    "burrito": ["burrito", "burritos"],
    "kebab": ["kebab"],
    "arepa": ["arepa", "arepas"],
    "coxinha": ["coxinha", "coxinhas"],
    "açai": ["açai", "acai"],
    "poke": ["poke", "poké"],
    # Desayunos/postres
    "tostada": ["tostada", "tostadas"],
    "churros": ["churros", "churros con chocolate"],
    "tarta de queso": ["tarta", "tarta de queso", "cheesecake"],
    "torrija": ["torrija", "torrijas"],
    "helado": ["helado", "helados"],
    "flan": ["flan"],
    "brownie": ["brownie"],
    "crepe": ["crepe", "crepes"],
    "gofre": ["gofre", "gofres"],
    "pastel": ["pastel", "pasteles"],
    "dulce": ["dulce", "dulces"],
    # Formatos de carta que pueden ser plato si van compuestos
    "montadito": ["montadito", "montaditos"],
    "bocadillo": ["bocadillo", "bocadillos"],
    "tosta": ["tosta", "tostas"],
    "empanada": ["empanada", "empanadas"],
    "ensalada": ["ensalada", "ensaladas"],
}

for canonical, aliases in DISH_ALIASES.items():
    for alias in aliases:
        add_alias(alias, canonical=canonical, group="dish_core", priority=1.0)

GENERIC_FOOD_TERMS = [
    "tapa", "tapas", "ración", "racion", "raciones", "plato", "platos", "comida", "cena", "almuerzo",
    "desayuno", "menú", "menu", "carta", "cocina", "entrante", "entrantes", "postre", "postres",
    "aperitivo", "aperitivos", "producto", "productos", "especialidad", "especialidades",
]
for term in GENERIC_FOOD_TERMS:
    add_alias(term, canonical=term, group="generic_food_term", priority=0.35)

BEVERAGE_TERMS = [
    "cerveza", "cervezas", "vino", "vinos", "copa", "copas", "café", "cafe", "cafés", "cafes",
    "refresco", "refrescos", "agua", "tinto", "tinto de verano", "vermut", "manzanilla", "rebujito",
    "mojito", "cóctel", "coctel", "cócteles", "cocteles", "sangría", "sangria", "batido", "batidos",
]
for term in BEVERAGE_TERMS:
    add_alias(term, canonical=term, group="beverage", priority=0.2)

INGREDIENT_OR_SIDE_TERMS = [
    "pan", "queso", "salsa", "carne", "pescado", "pollo", "ternera", "cerdo", "jamón", "jamon",
    "patatas", "papas", "arroz", "huevo", "huevos", "tomate", "setas", "verduras", "trufa",
    "chocolate", "nata", "miel", "alioli", "mayonesa", "cebolla", "pimientos", "marisco",
]
for term in INGREDIENT_OR_SIDE_TERMS:
    add_alias(term, canonical=term, group="ingredient_or_side", priority=0.45)

aliases_df = pd.DataFrame(alias_records).drop_duplicates(subset=["alias_norm", "group", "canonical"])

# Si un alias cae en varios grupos, priorizamos dish_core > ingredient > generic > beverage.
group_priority = {"dish_core": 4, "ingredient_or_side": 3, "generic_food_term": 2, "beverage": 1}
aliases_df["group_priority"] = aliases_df["group"].map(group_priority)
aliases_df = (
    aliases_df.sort_values(["alias_norm", "group_priority", "priority"], ascending=[True, False, False])
    .drop_duplicates(subset=["alias_norm"], keep="first")
    .sort_values("alias_norm")
    .reset_index(drop=True)
)

print("Aliases únicos:", len(aliases_df))
print("Distribución por grupo:")
print(aliases_df["group"].value_counts())
display(aliases_df.head(30))


Aliases únicos: 201
Distribución por grupo:
group
dish_core             131
ingredient_or_side     25
generic_food_term      23
beverage               22
Name: count, dtype: int64


,alias,alias_norm,canonical,group,priority,group_priority
0,açai,acai,acai,dish_core,1.00,4
1,adobo,adobo,cazon en adobo,dish_core,1.00,4
2,agua,agua,agua,beverage,0.20,1
3,albóndigas,albondigas,albondigas,dish_core,1.00,4
4,alioli,alioli,alioli,ingredient_or_side,0.45,3
5,almejas,almejas,almejas,dish_core,1.00,4
6,almuerzo,almuerzo,almuerzo,generic_food_term,0.35,2
7,aperitivo,aperitivo,aperitivo,generic_food_term,0.35,2
8,aperitivos,aperitivos,aperitivos,generic_food_term,0.35,2
9,arepa,arepa,arepa,dish_core,1.00,4


In [18]:
# ============================================================
# 06. Patrones de platos compuestos
# ============================================================

# Estos patrones capturan menciones más ricas que una palabra suelta:
# croquetas de jamón, solomillo al whisky, tarta de queso, montadito de pringá, etc.

DESCRIPTOR_TERMS = [
    "jamon iberico", "jamon", "puchero", "setas", "boletus", "queso de cabra", "queso", "pollo",
    "atun rojo", "atun", "bacalao", "gambas", "langostinos", "chocos", "calamares", "pulpo",
    "rabo de toro", "cola de toro", "carne mechada", "carne", "presa", "secreto", "iberico", "iberica",
    "pringa", "morcilla", "salmorejo", "tomate", "patatas", "papas", "verduras", "trufa", "miel",
    "whisky", "roquefort", "pimienta", "alioli", "ajo", "la abuela", "chocolate", "leche", "marisco",
]

# Se normalizan y se ordenan de más largo a más corto para que el regex prefiera expresiones compuestas.
descriptor_norms = sorted({normalize_text_value(x) for x in DESCRIPTOR_TERMS}, key=len, reverse=True)
DESCRIPTOR_ALT = "(?:" + "|".join(re.escape(x).replace("\ ", r"\s+") for x in descriptor_norms) + ")"

HEAD_TERMS = [
    "croqueta", "croquetas", "tortilla", "tarta", "tosta", "tostas", "tostada", "tostadas",
    "montadito", "montaditos", "bocadillo", "bocadillos", "hamburguesa", "hamburguesas",
    "pizza", "ensalada", "arroz", "pasta", "ravioli", "empanada", "arepa", "arepas", "taco", "tacos",
    "huevos", "solomillo", "lomo", "presa", "secreto", "carrillada", "bacalao", "atun",
    "pulpo", "gambas", "langostinos", "calamares", "chocos", "pollo", "pescado",
]
HEAD_ALT = "(?:" + "|".join(re.escape(normalize_text_value(x)) for x in sorted(HEAD_TERMS, key=len, reverse=True)) + ")"

COMPOUND_PATTERNS = [
    ("head_de_descriptor", re.compile(rf"(?<![a-z0-9_]){HEAD_ALT}\s+(?:de|del|con)\s+{DESCRIPTOR_ALT}(?![a-z0-9_])")),
    ("head_al_descriptor", re.compile(rf"(?<![a-z0-9_]){HEAD_ALT}\s+(?:al|a\s+la|a\s+los|a\s+las)\s+{DESCRIPTOR_ALT}(?![a-z0-9_])")),
    ("patatas_style", re.compile(r"(?<![a-z0-9_])(?:patatas|papas)\s+(?:bravas|alioli|fritas|arrugadas|gajo)(?![a-z0-9_])")),
    ("huevos_style", re.compile(r"(?<![a-z0-9_])huevos\s+(?:rotos|estrellados|revueltos)(?![a-z0-9_])")),
    ("pescado_style", re.compile(r"(?<![a-z0-9_])pescado\s+(?:frito|a\s+la\s+plancha)(?![a-z0-9_])")),
    ("solomillo_style", re.compile(r"(?<![a-z0-9_])(?:solomillo|lomo)\s+(?:al\s+whisky|al\s+roquefort|a\s+la\s+pimienta)(?![a-z0-9_])")),
    ("tarta_style", re.compile(r"(?<![a-z0-9_])tarta\s+(?:de\s+queso|de\s+chocolate|de\s+la\s+abuela)(?![a-z0-9_])")),
]

print("Patrones compuestos definidos:", [name for name, _ in COMPOUND_PATTERNS])


Patrones compuestos definidos: ['head_de_descriptor', 'head_al_descriptor', 'patatas_style', 'huevos_style', 'pescado_style', 'solomillo_style', 'tarta_style']


<>:18: SyntaxWarning: invalid escape sequence '\ '
<>:18: SyntaxWarning: invalid escape sequence '\ '
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_12284\2069402714.py:18: SyntaxWarning: invalid escape sequence '\ '
  DESCRIPTOR_ALT = "(?:" + "|".join(re.escape(x).replace("\ ", r"\s+") for x in descriptor_norms) + ")"


In [8]:
# ============================================================
# 07. Señales contextuales y reglas de confianza
# ============================================================

ORDER_TRIGGERS = [
    "pedimos", "pedi", "pedí", "pedir", "probamos", "probe", "probé", "probar", "tomamos",
    "comimos", "cenamos", "desayunamos", "recomiendo", "recomendamos", "recomendable",
    "nos pusieron", "nos sirvieron", "nos trajeron", "especialidad", "plato", "tapa", "racion", "ración",
]
POSITIVE_TERMS = [
    "rico", "riquisimo", "riquísimo", "bueno", "buenisimo", "buenísimo", "espectacular", "brutal",
    "increible", "increíble", "excelente", "delicioso", "deliciosa", "casero", "casera", "fresco", "fresca",
    "sabroso", "sabrosa", "perfecto", "perfecta", "repetir", "volveremos", "imprescindible",
]
NEGATIVE_TERMS = [
    "malo", "mala", "regular", "frio", "fría", "fria", "seco", "seca", "duro", "dura",
    "soso", "sosa", "salado", "salada", "quemado", "quemada", "decepcionante", "caro", "cara",
    "sin sabor", "no merece", "no volver", "horrible", "fatal",
]
CONTRAST_TERMS = ["pero", "aunque", "sin embargo", "no obstante", "eso si", "eso sí"]

ORDER_TRIGGERS_N = [normalize_text_value(x) for x in ORDER_TRIGGERS]
POSITIVE_TERMS_N = [normalize_text_value(x) for x in POSITIVE_TERMS]
NEGATIVE_TERMS_N = [normalize_text_value(x) for x in NEGATIVE_TERMS]
CONTRAST_TERMS_N = [normalize_text_value(x) for x in CONTRAST_TERMS]

CANDIDATE_TYPE_PRIORITY = {
    "dish_candidate": 5,
    "uncertain": 4,
    "ingredient_or_side": 3,
    "generic_food_term": 2,
    "beverage": 1,
    "excluded": 0,
}

def contains_any(context_norm: str, terms) -> bool:
    return any(term in context_norm for term in terms)

def context_signals(text_match: str, start: int, end: int, window: int = 90):
    left = max(0, start - window)
    right = min(len(text_match), end + window)
    context = text_match[left:right]
    return {
        "has_order_trigger": contains_any(context, ORDER_TRIGGERS_N),
        "has_positive_context": contains_any(context, POSITIVE_TERMS_N),
        "has_negative_context": contains_any(context, NEGATIVE_TERMS_N),
        "has_contrast_context": contains_any(context, CONTRAST_TERMS_N),
    }

def base_candidate_type(group: str, mention_norm: str, detection_method: str) -> str:
    if group == "dish_core":
        return "dish_candidate"
    if group == "beverage":
        return "beverage"
    if group == "generic_food_term":
        return "generic_food_term"
    if group == "ingredient_or_side":
        return "ingredient_or_side"
    if detection_method.startswith("compound"):
        return "dish_candidate"
    return "uncertain"

def compute_confidence(candidate_type: str, detection_method: str, mention_norm: str, signals: dict) -> float:
    if detection_method.startswith("compound"):
        score = 0.90
    elif candidate_type == "dish_candidate":
        # Una palabra suelta como "tarta" o "tortilla" es algo menos segura que "tarta de queso".
        score = 0.80 if " " not in mention_norm else 0.88
    elif candidate_type == "ingredient_or_side":
        score = 0.52
    elif candidate_type == "generic_food_term":
        score = 0.45
    elif candidate_type == "beverage":
        score = 0.35
    else:
        score = 0.50

    if signals.get("has_order_trigger"):
        score += 0.05
    if signals.get("has_positive_context") or signals.get("has_negative_context"):
        score += 0.03
    if signals.get("has_contrast_context"):
        score -= 0.02

    # Penalizaciones intencionadas.
    if candidate_type in {"generic_food_term", "beverage"}:
        score -= 0.05

    return round(max(0.05, min(0.99, score)), 4)


In [ ]:
# ============================================================
# 08. Compilación de aliases y resolución de solapes
# ============================================================

def alias_to_regex(alias_norm: str) -> re.Pattern:
    escaped = re.escape(alias_norm)..replace("\\ ", r"\s+"))
    return re.compile(rf"(?<![a-z0-9_]){escaped}(?![a-z0-9_])")

compiled_aliases = []
for row in aliases_df.to_dict("records"):
    compiled_aliases.append({
        **row,
        "regex": alias_to_regex(row["alias_norm"]),
        "alias_length": len(row["alias_norm"]),
    })

# Priorizar aliases más largos y de grupo más importante.
compiled_aliases = sorted(
    compiled_aliases,
    key=lambda x: (group_priority.get(x["group"], 0), x["alias_length"]),
    reverse=True,
)


def spans_overlap(a, b) -> bool:
    return not (a["end_char"] <= b["start_char"] or b["end_char"] <= a["start_char"])


def resolve_overlaps(candidates):
    """Conserva la mejor mención cuando varios candidatos se pisan."""
    ordered = sorted(
        candidates,
        key=lambda c: (
            CANDIDATE_TYPE_PRIORITY.get(c["candidate_type"], 0),
            c["confidence"],
            c["end_char"] - c["start_char"],
        ),
        reverse=True,
    )
    selected = []
    for cand in ordered:
        if not any(spans_overlap(cand, prev) for prev in selected):
            selected.append(cand)
    return sorted(selected, key=lambda c: (c["start_char"], c["end_char"]))


<>:6: SyntaxWarning: invalid escape sequence '\ '
<>:6: SyntaxWarning: invalid escape sequence '\ '
C:\Users\USUARIO\AppData\Local\Temp\ipykernel_12284\2555315659.py:6: SyntaxWarning: invalid escape sequence '\ '
  escaped = re.escape(alias_norm).replace("\ ", r"\s+")


In [10]:
# ============================================================
# 09. Extracción de candidatos por review
# ============================================================

def canonical_from_phrase(mention_norm: str) -> str:
    """Normalización inicial de nombre canónico. La normalización fina vendrá en el notebook 14."""
    mention_norm = compact_spaces(mention_norm.lower())
    # Reglas directas para variantes muy frecuentes.
    direct = {
        "bravas": "patatas bravas",
        "papas bravas": "patatas bravas",
        "croquetas": "croqueta",
        "croquetas caseras": "croqueta",
        "tortilla": "tortilla de patatas",
        "tarta": "tarta de queso",
        "atun": "atun",
        "atún": "atun",
        "carrilleras": "carrillada",
        "carrillera": "carrillada",
        "hamburguesas": "hamburguesa",
        "tostadas": "tostada",
        "montaditos": "montadito",
        "bocadillos": "bocadillo",
    }
    if mention_norm in direct:
        return direct[mention_norm]
    return mention_norm


def build_candidate(row, start, end, mention_text, mention_norm, canonical, group, detection_method, pattern_name=None):
    text = row["text_clean"]
    text_match = row["text_match"]
    signals = context_signals(text_match, start, end)
    ctype = base_candidate_type(group, mention_norm, detection_method)
    confidence = compute_confidence(ctype, detection_method, mention_norm, signals)

    return {
        "mention_id": stable_id(row["review_id"], start, end, mention_norm, detection_method),
        "review_id": row["review_id"],
        "place_id": row["place_id"],
        "place_source_ref_id": row.get("place_source_ref_id"),
        "source_review_id": row.get("source_review_id"),
        "source_place_record_id": row.get("source_place_record_id"),
        "place_name": row.get("place_name"),
        "district_id": row.get("district_id"),
        "district_name": row.get("district_name"),
        "neighborhood_id": row.get("neighborhood_id"),
        "neighborhood_name": row.get("neighborhood_name"),
        "rating_value": row.get("rating_value"),
        "review_sentiment_from_rating": row.get("review_sentiment_from_rating"),
        "review_language": row.get("review_language"),
        "dish_mention_text": compact_spaces(mention_text),
        "dish_mention_normalized": compact_spaces(mention_norm),
        "canonical_dish_name_es_v1": canonical_from_phrase(canonical),
        "candidate_type": ctype,
        "lexicon_group": group,
        "detection_method": detection_method,
        "pattern_name": pattern_name,
        "confidence": confidence,
        "start_char": int(start),
        "end_char": int(end),
        "context_sentence": get_sentence_context(text, start, end),
        "window_context": get_window_context(text, start, end, window=140),
        "has_order_trigger": signals["has_order_trigger"],
        "has_positive_context": signals["has_positive_context"],
        "has_negative_context": signals["has_negative_context"],
        "has_contrast_context": signals["has_contrast_context"],
        "text": text,
    }


def extract_candidates_from_review(row):
    text = row["text_clean"]
    text_match = row["text_match"]
    candidates = []

    # 1) Aliases exactos.
    for item in compiled_aliases:
        for m in item["regex"].finditer(text_match):
            start, end = m.span()
            mention_text = text[start:end]
            mention_norm = normalize_text_value(mention_text)
            candidates.append(build_candidate(
                row=row,
                start=start,
                end=end,
                mention_text=mention_text,
                mention_norm=mention_norm,
                canonical=item["canonical"],
                group=item["group"],
                detection_method="exact_alias",
                pattern_name=None,
            ))

    # 2) Patrones compuestos.
    for pattern_name, pattern in COMPOUND_PATTERNS:
        for m in pattern.finditer(text_match):
            start, end = m.span()
            mention_text = text[start:end]
            mention_norm = normalize_text_value(mention_text)
            candidates.append(build_candidate(
                row=row,
                start=start,
                end=end,
                mention_text=mention_text,
                mention_norm=mention_norm,
                canonical=mention_norm,
                group="dish_core",
                detection_method="compound_pattern",
                pattern_name=pattern_name,
            ))

    # 3) Resolver solapes y devolver.
    return resolve_overlaps(candidates)

# Preparar texto normalizado para matching.
df["text_match"] = df["text_clean"].apply(normalize_for_match_keep_len)

# Smoke test con una review manual.
manual_row = {
    "review_id": "manual_1",
    "place_id": "place_1",
    "place_source_ref_id": "psr_1",
    "source_review_id": "source_review_1",
    "source_place_record_id": "source_place_1",
    "place_name": "Demo",
    "district_id": None,
    "district_name": "Demo",
    "neighborhood_id": None,
    "neighborhood_name": "Demo",
    "rating_value": 5,
    "review_sentiment_from_rating": "positive",
    "review_language": "es",
    "text_clean": "Pedimos croquetas de jamón, solomillo al whisky, patatas bravas y una cerveza.",
}
manual_row["text_match"] = normalize_for_match_keep_len(manual_row["text_clean"])
manual_candidates = extract_candidates_from_review(manual_row)
pd.DataFrame(manual_candidates)[["dish_mention_text", "canonical_dish_name_es_v1", "candidate_type", "confidence", "detection_method"]]


,dish_mention_text,canonical_dish_name_es_v1,candidate_type,confidence,detection_method
0,croquetas de jamón,croquetas de jamon,dish_candidate,0.95,compound_pattern
1,solomillo al whisky,solomillo al whisky,dish_candidate,0.95,compound_pattern
2,patatas bravas,patatas bravas,dish_candidate,0.95,compound_pattern
3,cerveza,cerveza,beverage,0.35,exact_alias


In [11]:
# ============================================================
# 10. Ejecutar detección sobre todo el corpus
# ============================================================

all_candidates = []
for row in df.to_dict("records"):
    all_candidates.extend(extract_candidates_from_review(row))

candidates_df = pd.DataFrame(all_candidates)

print("Candidatos detectados:", len(candidates_df))
if len(candidates_df):
    print("Reviews con candidatos:", candidates_df["review_id"].nunique())
    print("Locales con candidatos:", candidates_df["place_id"].nunique())
    print("Distribución candidate_type:")
    print(candidates_df["candidate_type"].value_counts())
    display(candidates_df.head(20))
else:
    print("No se detectaron candidatos. Revisar lexicón/patrones.")


Candidatos detectados: 12854
Reviews con candidatos: 3681
Locales con candidatos: 825
Distribución candidate_type:
candidate_type
generic_food_term     5999
dish_candidate        3659
ingredient_or_side    1944
beverage              1252
Name: count, dtype: int64


,mention_id,review_id,place_id,place_source_ref_id,source_review_id,source_place_record_id,place_name,district_id,district_name,neighborhood_id,neighborhood_name,rating_value,review_sentiment_from_rating,review_language,dish_mention_text,dish_mention_normalized,canonical_dish_name_es_v1,candidate_type,lexicon_group,detection_method,pattern_name,confidence,start_char,end_char,context_sentence,window_context,has_order_trigger,has_positive_context,has_negative_context,has_contrast_context,text
0,1fde28a5f80c28e3948ba1bf865c0b74017762f5,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5.0,positive,es,Açai,acai,acai,dish_candidate,dish_core,exact_alias,NaN,0.83,19,23,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedaci",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis ami..."
1,b70f05c16925cdb70c7f0ea3f4ce0d328d3ccc24,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,2acb2538-7f10-4760-a646-85c41436f9e8,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed81730074b12931536495c,ChIJ50jgAn5tEg0RBySkaJ7I8eE,Tropitali,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5.0,positive,es,coxinha,coxinha,coxinha,dish_candidate,dish_core,exact_alias,NaN,0.83,59,66,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables.","Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré",False,True,False,False,"Sin dudas el mejor Açai que he probado aqui en Sevilla, la coxinha muy rica y el trato del personal increíble, son muy muy muy amables. Gracias por traer un pedacito de Brasil ❤️🇧🇷 volveré y seguro indicaré a mis ami..."
2,d2e53ecfefa1be6efcb77e2ab76c4c6274308211,b680ad6e-8999-48a0-a754-84ea920429ab,a6ba370c-eb3d-4456-94cf-58617d1a3b0a,9f4c2eb7-2ff7-44ce-8db2-1be259e6985d,4897b8c24f949dda32b4e6c5cad74f4a91228c64182c5ce9408158505e192ce0,ChIJuQj13EFvEg0RPkUdEUzL0BA,Cariña VZ,22b433e4-3b14-4b09-bfaa-f0ca1d7e6bdc,Sur,65af0982-2885-49df-9f09-8b784c2f3d7b,GIRALDA SUR,5.0,positive,es,comida,comida,comida,generic_food_term,generic_food_term,exact_alias,NaN,0.43,6,12,"Buena comida y con un excelente servicio de Leo que nos atendió y nos explicó todo muy bien, sin duda iremos más de una vez.","Buena comida y con un excelente servicio de Leo que nos atendió y nos explicó todo muy bien, sin duda iremos más de una vez.",False,True,False,False,"Buena comida y con un excelente servicio de Leo que nos atendió y nos explicó todo muy bien, sin duda iremos más de una vez."
3,c0d9d8a473412c03003e6346dae49d8eef4c322f,80af93bc-d556-46a4-8254-485a921f7ac2,7d39aad6-1d3f-4fc8-8d4b-22896354507e,4fa65ed3-e51a-4a96-b82c-cbe002c03141,a74b0bb34a2b817d7ead9adbb12509984985802fe0be90942815bf3c5f52db00,ChIJmU6FwYdvEg0RYLF2daPZQkc,TOTORA,09b2875d-a826-4c21-bd78-1d8fa48009de,Cerro - Amate,76a474aa-a390-48b4-a7ad-d68f6d747832,LA PLATA,2.0,negative,es,platos,platos,platos,generic_food_term,generic_food_term,exact_alias,NaN,0.45,76,82,"Una decepción, esperé 2 horas mi pedido, la camarera viendo que salían otro platos de personas que recién llegaban y no tuvo la delicadeza de ver que no llegaba el nuestro , 

In [12]:
# ============================================================
# 11. Separación de candidatos útiles para plato
# ============================================================

if len(candidates_df):
    # Candidatos fuertes para pasar a normalización posterior.
    dish_candidates_df = candidates_df[
        (candidates_df["candidate_type"] == "dish_candidate")
        & (candidates_df["confidence"] >= 0.70)
    ].copy()

    # Candidatos ampliados para revisión manual.
    reviewable_candidates_df = candidates_df[
        candidates_df["candidate_type"].isin(["dish_candidate", "uncertain", "ingredient_or_side", "generic_food_term"])
    ].copy()

    print("Dish candidates v1:", len(dish_candidates_df))
    print("Reviews con dish candidates:", dish_candidates_df["review_id"].nunique())
    print("Locales con dish candidates:", dish_candidates_df["place_id"].nunique())
    print("Candidatos revisables:", len(reviewable_candidates_df))

    display(
        dish_candidates_df[
            [
                "dish_mention_text", "canonical_dish_name_es_v1", "candidate_type", "confidence",
                "place_name", "district_name", "neighborhood_name", "rating_value", "context_sentence"
            ]
        ].sort_values(["confidence", "canonical_dish_name_es_v1"], ascending=[False, True]).head(40)
    )
else:
    dish_candidates_df = pd.DataFrame()
    reviewable_candidates_df = pd.DataFrame()


Dish candidates v1: 3659
Reviews con dish candidates: 1738
Locales con dish candidates: 706
Candidatos revisables: 11602


,dish_mention_text,canonical_dish_name_es_v1,candidate_type,confidence,place_name,district_name,neighborhood_name,rating_value,context_sentence
4164,arroz con cola de toro,arroz con cola de toro,dish_candidate,0.98,Bar El 25 Bodega,Los Remedios,LOS REMEDIOS,5.0,"El san Jacobo enorme y tierno, los calamares del campo buenísimos, arroz con cola de toro, las migas espectaculares..."
3392,arroz con leche,arroz con leche,dish_candidate,0.98,Restaurante Lalola de Javi Abascal,Casco Antiguo,FERIA,5.0,"Comida estupenda y un trato exquisito, la ensaladilla muy buena, los espárragos con setas y jabalí, luego pedimos una corvina al ajillo y terminamos con un arroz con leche con torreznos caramelizado , todo exquisito ..."
4977,arroz con leche,arroz con leche,dish_candidate,0.98,Doña Pepa,Macarena,CRUZ ROJA-CAPUCHINOS,5.0,Luna el camarero que nos atendió muy cortes y educado la comida calientita buena porción de tapas y chupito de arroz con leche riquísimo 🤤
7442,arroz con leche,arroz con leche,dish_candidate,0.98,"La Parrala Bar&Tapas. Triana, Sevilla",Triana,TRIANA CASCO ANTIGUO,4.0,Al terminar nos pusieron un chupito de licor de arroz con leche!!
6860,Arroz con rabo de toro,arroz con rabo de toro,dish_candidate,0.98,Antigua Casa Diego,Triana,TRIANA CASCO ANTIGUO,4.0,-Arroz con rabo de toro...
6629,arroz de boletus,arroz de boletus,dish_candidate,0.98,Restaurante Monte Tradición,San Pablo - Santa Justa,SANTA CLARA,5.0,Ambiente y atención sobresaliente 4 de nosotros hemos comido un arroz de boletus y Angus y un chuletón de carne madura.
296,arroz de marisco,arroz de marisco,dish_candidate,0.98,Doña Encarna,Casco Antiguo,ALFALFA,5.0,Pedimos el arroz de marisco y los buñuelos de bacalao.
5195,carrillada con queso de cabra,carrillada con queso de cabra,dish_candidate,0.98,"Mesón ""La Madre""",Los Remedios,ISLA DE GARZA,4.0,"La comida espectacular, los dos arroces que probamos con mariscos el primero y carrillada con queso de cabra excelentes , los entrantes unas croquetas de pucheros muy suaves y cremosas con muy buen sabor, la tabla co..."
8364,carrillada con queso de cabra,carrillada con queso de cabra,dish_candidate,0.98,La Tontona Abacería,Bellavista - La Palmera,BELLAVISTA,5.0,"Probamos las tostas de carrillada con queso de cabra, los chicharrones de Cádiz, la de salmón… todo estaba genial!"
4140,carrillada con setas,carrillada con setas,dish_candidate,0.98,Bar Cuando Puedas,Macarena,LOS PRINCIPES-LA FONTANILLA,5.0,"Todo espectacular de bueno, toda la sorpresa, comimos Mojama, ensaladilla de pulpo, pavia y gyozas de carrillada con setas…."


In [13]:
# ============================================================
# 12. Resúmenes de candidatos
# ============================================================

if len(candidates_df):
    candidate_type_summary = (
        candidates_df.groupby("candidate_type")
        .agg(
            mention_count=("mention_id", "count"),
            review_count=("review_id", "nunique"),
            place_count=("place_id", "nunique"),
            avg_confidence=("confidence", "mean"),
        )
        .reset_index()
        .sort_values("mention_count", ascending=False)
    )

    top_canonical_dishes = (
        dish_candidates_df.groupby("canonical_dish_name_es_v1")
        .agg(
            mention_count=("mention_id", "count"),
            review_count=("review_id", "nunique"),
            place_count=("place_id", "nunique"),
            avg_confidence=("confidence", "mean"),
            avg_rating=("rating_value", "mean"),
        )
        .reset_index()
        .sort_values(["mention_count", "place_count"], ascending=False)
    )

    candidates_by_district = (
        dish_candidates_df.groupby(["district_name", "canonical_dish_name_es_v1"])
        .agg(
            mention_count=("mention_id", "count"),
            review_count=("review_id", "nunique"),
            place_count=("place_id", "nunique"),
            avg_rating=("rating_value", "mean"),
        )
        .reset_index()
        .sort_values(["district_name", "mention_count"], ascending=[True, False])
    )

    candidates_by_neighborhood = (
        dish_candidates_df.groupby(["district_name", "neighborhood_name", "canonical_dish_name_es_v1"])
        .agg(
            mention_count=("mention_id", "count"),
            review_count=("review_id", "nunique"),
            place_count=("place_id", "nunique"),
            avg_rating=("rating_value", "mean"),
        )
        .reset_index()
        .sort_values(["district_name", "neighborhood_name", "mention_count"], ascending=[True, True, False])
    )

    print("Resumen por tipo:")
    display(candidate_type_summary)
    print("Top platos canónicos v1:")
    display(top_canonical_dishes.head(50))
else:
    candidate_type_summary = pd.DataFrame()
    top_canonical_dishes = pd.DataFrame()
    candidates_by_district = pd.DataFrame()
    candidates_by_neighborhood = pd.DataFrame()


Resumen por tipo:


,candidate_type,mention_count,review_count,place_count,avg_confidence
2,generic_food_term,5999,2935,800,0.443981
1,dish_candidate,3659,1738,706,0.859874
3,ingredient_or_side,1944,1088,601,0.559249
0,beverage,1252,904,492,0.330575


Top platos canónicos v1:


,canonical_dish_name_es_v1,mention_count,review_count,place_count,avg_confidence,avg_rating
68,ensaladilla,216,197,153,0.846528,4.180556
40,croqueta,211,198,160,0.852938,4.312796
171,tostada,209,167,119,0.830000,3.794258
29,carrillada,172,152,123,0.850756,4.488372
146,solomillo al whisky,167,152,126,0.880958,4.209581
4,arroz,153,132,109,0.851895,4.169935
19,atun,117,99,70,0.851111,4.401709
157,tarta de queso,114,100,81,0.892018,4.473684
96,montadito,108,95,73,0.842593,4.398148
116,patatas bravas,105,98,76,0.912095,4.161905


In [14]:
# ============================================================
# 13. Muestras para revisión manual
# ============================================================

if len(candidates_df):
    # 1) Muestra prioritaria: candidatos de plato con más confianza y contexto.
    manual_top_dish = dish_candidates_df.sort_values(
        ["confidence", "canonical_dish_name_es_v1", "rating_value"],
        ascending=[False, True, False]
    ).head(500)

    # 2) Muestra de casos dudosos/generic/ingredient para decidir reglas de exclusión.
    manual_uncertain = reviewable_candidates_df[
        reviewable_candidates_df["candidate_type"] != "dish_candidate"
    ].sort_values(["candidate_type", "confidence"], ascending=[True, False]).head(500)

    manual_review_df = pd.concat([manual_top_dish, manual_uncertain], ignore_index=True)

    manual_cols = [
        "candidate_type", "dish_mention_text", "canonical_dish_name_es_v1", "confidence", "detection_method", "pattern_name",
        "place_name", "district_name", "neighborhood_name", "rating_value", "review_sentiment_from_rating",
        "has_order_trigger", "has_positive_context", "has_negative_context", "has_contrast_context",
        "context_sentence", "window_context", "review_id", "place_id", "mention_id",
    ]
    manual_cols = [c for c in manual_cols if c in manual_review_df.columns]
    manual_review_df = manual_review_df[manual_cols]

    display(manual_review_df.head(50))
else:
    manual_review_df = pd.DataFrame()


,candidate_type,dish_mention_text,canonical_dish_name_es_v1,confidence,detection_method,pattern_name,place_name,district_name,neighborhood_name,rating_value,review_sentiment_from_rating,has_order_trigger,has_positive_context,has_negative_context,has_contrast_context,context_sentence,window_context,review_id,place_id,mention_id
0,dish_candidate,arroz con cola de toro,arroz con cola de toro,0.98,compound_pattern,head_de_descriptor,Bar El 25 Bodega,Los Remedios,LOS REMEDIOS,5.0,positive,True,True,False,False,"El san Jacobo enorme y tierno, los calamares del campo buenísimos, arroz con cola de toro, las migas espectaculares...","es. Incluso las patatas fritas que acompañan algunos platos son caseras. El san Jacobo enorme y tierno, los calamares del campo buenísimos, arroz con cola de toro, las migas espectaculares... Todo buenísimo. El trato...",eec2d563-3a50-49ca-8445-948e4df86ea1,e62bfa58-5506-4a06-8767-27d5c18ee68a,07f2f5464379efe4bc54d40e50aa0a86ed9af15c
1,dish_candidate,arroz con leche,arroz con leche,0.98,compound_pattern,head_de_descriptor,Restaurante Lalola de Javi Abascal,Casco Antiguo,FERIA,5.0,positive,True,False,True,False,"Comida estupenda y un trato exquisito, la ensaladilla muy buena, los espárragos con setas y jabalí, luego pedimos una corvina al ajillo y terminamos con un arroz con leche con torreznos caramelizado , todo exquisito ...","y un trato exquisito, la ensaladilla muy buena, los espárragos con setas y jabalí, luego pedimos una corvina al ajillo y terminamos con un arroz con leche con torreznos caramelizado , todo exquisito , muy recomendable",1541ca78-01be-4be2-a4dd-b140c7cd2e77,9d5876f8-a61f-4c94-8b42-f511dd44930c,023e06ff77f5d7aa523d2104dd0066528c518ea2
2,dish_candidate,arroz con leche,arroz con leche,0.98,compound_pattern,head_de_descriptor,Doña Pepa,Macarena,CRUZ ROJA-CAPUCHINOS,5.0,positive,True,True,False,False,Luna el camarero que nos atendió muy cortes y educado la comida calientita buena porción de tapas y chupito de arroz con leche riquísimo 🤤,impio y muy bonito decorado! Luna el camarero que nos atendió muy cortes y educado la comida calientita buena porción de tapas y chupito de arroz con leche riquísimo 🤤,aeed265b-6de1-432d-bd98-8586dbd82694,d59bcf32-9b47-4a64-a04d-a16779a81f46,c710c6c4f227e27fb08400e4c2c4486f18ae6073
3,dish_candidate,arroz con leche,arroz con leche,0.98,compound_pattern,head_de_descriptor,"La Parrala Bar&Tapas. Triana, Sevilla",Triana,TRIANA CASCO ANTIGUO,4.0,positive,True,True,False,False,Al terminar nos pusieron un chupito de licor de arroz con leche!!,"dos los platos a la vez, eso podrían mejorarlo para que no se acumule (y enfríe) la comida. Al terminar nos pusieron un chupito de licor de arroz con leche!! Repetiré!!",714b93dc-cdbc-47c1-b291-c5d278a02c30,08374e09-2b07-424c-b290-3761827a725d,528c29ecd3b9dc2f6fe2a9f47bb532cb6ac491b1
4,dish_candidate,Arroz con rabo de toro,arroz con rabo de toro,0.98,compound_pattern,head_de_descriptor,Antigua Casa Diego,Triana,TRIANA CASCO ANTIGUO,4.0,positive,True,True,False,False,-Arroz con rabo de toro...,"ado para otro intentando llegar a todo...o sea FALTA PERSONAL porque van sin aliento. Pedimos: -Guiso de espinacas muy rico, algo picante. -Arroz con rabo de toro...muy flojito -Ensaladilla de pulpo, rica, correcta. ...",4eb2a628-a5be-4e89-bbf8-8a299385b4be,e868e012-6e0d-4782-8882-8ddb60dc060c,37433da838d77d7afef7abc7e21d2bb6212b8035
5,dish_candidate,arroz de boletus,arroz de boletus,0.98,compound_pattern,head_de_descriptor,Restaurante Monte Tradición,San Pablo - Santa Justa,SANTA CLARA,5.0,positive,True,False,True,False,Ambiente y atención sobresaliente 4 de nosotros hemos comido un arroz de boletus y Angus y un chuletón de carne madura.,s venido a jugar un torneo de pádel 13 personas y hemos cenado aquí un día. Ambiente y atención sobresaliente 4 de nosotros hemos comido un arroz de boletus y Angus y un chuletón de carne madura. Mientras escribo est...,61352342-d76f-42c0-806b-24ce6eb5291a,26d7bd1c-85c7-4c8e-b3b4-55fc386784e6,331b157

In [15]:
# ============================================================
# 14. Exportar artefactos v1
# ============================================================


def write_jsonl(df_to_write: pd.DataFrame, path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as f:
        for row in df_to_write.to_dict("records"):
            f.write(json.dumps(row, ensure_ascii=False, default=str) + "\n")

all_candidates_csv = OUTPUT_DIR / "sevilla_dish_candidates_all_v1.csv"
all_candidates_jsonl = OUTPUT_DIR / "sevilla_dish_candidates_all_v1.jsonl"
dish_candidates_csv = OUTPUT_DIR / "sevilla_dish_candidates_v1.csv"
dish_candidates_jsonl = OUTPUT_DIR / "sevilla_dish_candidates_v1.jsonl"
manual_review_csv = OUTPUT_DIR / "sevilla_dish_candidates_for_manual_review_v1.csv"
canonical_summary_csv = OUTPUT_DIR / "sevilla_top_canonical_dishes_v1.csv"
district_summary_csv = OUTPUT_DIR / "sevilla_dish_candidates_by_district_v1.csv"
neighborhood_summary_csv = OUTPUT_DIR / "sevilla_dish_candidates_by_neighborhood_v1.csv"
type_summary_csv = OUTPUT_DIR / "sevilla_dish_candidate_type_summary_v1.csv"

if len(candidates_df):
    candidates_df.to_csv(all_candidates_csv, index=False, encoding="utf-8")
    write_jsonl(candidates_df, all_candidates_jsonl)

    dish_candidates_df.to_csv(dish_candidates_csv, index=False, encoding="utf-8")
    write_jsonl(dish_candidates_df, dish_candidates_jsonl)

    manual_review_df.to_csv(manual_review_csv, index=False, encoding="utf-8")
    top_canonical_dishes.to_csv(canonical_summary_csv, index=False, encoding="utf-8")
    candidates_by_district.to_csv(district_summary_csv, index=False, encoding="utf-8")
    candidates_by_neighborhood.to_csv(neighborhood_summary_csv, index=False, encoding="utf-8")
    candidate_type_summary.to_csv(type_summary_csv, index=False, encoding="utf-8")

    print("Artefactos generados en:", OUTPUT_DIR)
    for path in [
        all_candidates_csv,
        all_candidates_jsonl,
        dish_candidates_csv,
        dish_candidates_jsonl,
        manual_review_csv,
        canonical_summary_csv,
        district_summary_csv,
        neighborhood_summary_csv,
        type_summary_csv,
    ]:
        print("-", path)
else:
    print("No se exportan candidatos porque no hay resultados.")


Artefactos generados en: C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_detection
- C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_detection\sevilla_dish_candidates_all_v1.csv
- C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_detection\sevilla_dish_candidates_all_v1.jsonl
- C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_detection\sevilla_dish_candidates_v1.csv
- C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_detection\sevilla_dish_candidates_v1.jsonl
- C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_detection\sevilla_dish_candidates_for_manual_review_v1.csv
- C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pi

In [16]:
# ============================================================
# 15. Summary JSON del notebook
# ============================================================

summary = {
    "notebook": "13_sevilla_dish_candidate_detection",
    "version": RUN_NAME,
    "generated_at": GENERATED_AT,
    "input_path": str(INPUT_PATH),
    "output_dir": str(OUTPUT_DIR),
    "input": {
        "total_reviews": int(len(df)),
        "unique_places": int(df["place_id"].nunique()),
        "unique_neighborhoods": int(df["neighborhood_id"].nunique()),
        "unique_districts": int(df["district_id"].nunique()),
        "language_counts": {str(k): int(v) for k, v in df["review_language"].value_counts(dropna=False).to_dict().items()},
        "rating_counts": {str(k): int(v) for k, v in df["rating_value"].value_counts(dropna=False).sort_index().to_dict().items()},
    },
    "detection": {
        "total_candidates_all_types": int(len(candidates_df)),
        "total_dish_candidates": int(len(dish_candidates_df)),
        "reviews_with_any_candidate": int(candidates_df["review_id"].nunique()) if len(candidates_df) else 0,
        "reviews_with_dish_candidate": int(dish_candidates_df["review_id"].nunique()) if len(dish_candidates_df) else 0,
        "places_with_dish_candidate": int(dish_candidates_df["place_id"].nunique()) if len(dish_candidates_df) else 0,
        "candidate_type_counts": {str(k): int(v) for k, v in candidates_df["candidate_type"].value_counts().to_dict().items()} if len(candidates_df) else {},
        "detection_method_counts": {str(k): int(v) for k, v in candidates_df["detection_method"].value_counts().to_dict().items()} if len(candidates_df) else {},
    },
    "top_canonical_dishes": top_canonical_dishes.head(50).to_dict("records") if len(top_canonical_dishes) else [],
    "checks": {
        "has_candidates": bool(len(candidates_df) > 0),
        "has_dish_candidates": bool(len(dish_candidates_df) > 0),
        "candidate_ids_are_unique": bool(candidates_df["mention_id"].is_unique) if len(candidates_df) else True,
        "dish_candidate_ids_are_unique": bool(dish_candidates_df["mention_id"].is_unique) if len(dish_candidates_df) else True,
        "all_dish_candidates_have_place": bool(dish_candidates_df["place_id"].notna().all()) if len(dish_candidates_df) else True,
        "all_dish_candidates_have_review": bool(dish_candidates_df["review_id"].notna().all()) if len(dish_candidates_df) else True,
    },
    "artifacts": {
        "all_candidates_csv": all_candidates_csv.name,
        "all_candidates_jsonl": all_candidates_jsonl.name,
        "dish_candidates_csv": dish_candidates_csv.name,
        "dish_candidates_jsonl": dish_candidates_jsonl.name,
        "manual_review_csv": manual_review_csv.name,
        "canonical_summary_csv": canonical_summary_csv.name,
        "district_summary_csv": district_summary_csv.name,
        "neighborhood_summary_csv": neighborhood_summary_csv.name,
        "type_summary_csv": type_summary_csv.name,
    },
}

summary_path = OUTPUT_DIR / "sevilla_dish_detection_summary_v1.json"
with summary_path.open("w", encoding="utf-8") as f:
    json.dump(summary, f, ensure_ascii=False, indent=2, default=str)

print("Summary guardado en:", summary_path)
print(json.dumps(summary["detection"], ensure_ascii=False, indent=2))
print(json.dumps(summary["checks"], ensure_ascii=False, indent=2))


Summary guardado en: C:\Users\USUARIO\Documents\Proyectos_Master_IA_Big_Data\hidden-gems-pipeline\data\artifacts\ai\sevilla\dish_detection\sevilla_dish_detection_summary_v1.json
{
  "total_candidates_all_types": 12854,
  "total_dish_candidates": 3659,
  "reviews_with_any_candidate": 3681,
  "reviews_with_dish_candidate": 1738,
  "places_with_dish_candidate": 706,
  "candidate_type_counts": {
    "generic_food_term": 5999,
    "dish_candidate": 3659,
    "ingredient_or_side": 1944,
    "beverage": 1252
  },
  "detection_method_counts": {
    "exact_alias": 12352,
    "compound_pattern": 502
  }
}
{
  "has_candidates": true,
  "has_dish_candidates": true,
  "candidate_ids_are_unique": true,
  "dish_candidate_ids_are_unique": true,
  "all_dish_candidates_have_place": true,
  "all_dish_candidates_have_review": true
}


## 16. Interpretación y próximos pasos

Este notebook genera candidatos de platos con una estrategia híbrida inicial. La salida no debe considerarse todavía el ranking final.

Los siguientes pasos recomendados son:

1. Revisar `sevilla_dish_candidates_for_manual_review_v1.csv`.
2. Ajustar el lexicón y exclusiones si aparecen falsos positivos.
3. Crear el notebook `14_sevilla_dish_normalization.ipynb`.
4. Construir un catálogo español de platos y aliases.
5. Pasar después a sentimiento por mención y señales `place-dish`.

Criterio recomendado para avanzar:

```text
- suficientes candidatos `dish_candidate`;
- falsos positivos controlables;
- buena cobertura por local/barrio;
- términos genéricos y bebidas separados del ranking.
```
